In [1]:
import yfinance as yf
import pandas as pd
import numpy as np
from datetime import date, datetime, timedelta
# web scrapping
import bs4 as bs
import requests
import lxml
from functools import reduce
# matplotlib
import matplotlib.pyplot as plt
import seaborn as sns
import networkx as nx
from ipysigma import Sigma
from pyvis.network import Network
import requests
from bs4 import BeautifulSoup
from io import StringIO
from dbconnection import MySQLDatabase
from utils import getSymbols, getData, get_last_date, get_marketid_simbols
import warnings

warnings.filterwarnings("ignore", category=UserWarning, module="pandas")
sns.set_theme()

In [2]:
db = MySQLDatabase("financialmarkets")

## Se obtienen símbolos de wikipedia

In [9]:
df = getSymbols('https://en.wikipedia.org/wiki/List_of_S%26P_500_companies')
df

,Symbol,Security,GICS Sector,GICS Sub-Industry,Headquarters Location,Date added,CIK,Founded
0,MMM,3M,Industrials,Industrial Conglomerates,"Saint Paul, Minnesota",1957-03-04,66740,1902
1,AOS,A. O. Smith,Industrials,Building Products,"Milwaukee, Wisconsin",2017-07-26,91142,1916
2,ABT,Abbott Laboratories,Health Care,Health Care Equipment,"North Chicago, Illinois",1957-03-04,1800,1888
3,ABBV,AbbVie,Health Care,Biotechnology,"North Chicago, Illinois",2012-12-31,1551152,2013 (1888)
4,ACN,Accenture,Information Technology,IT Consulting & Other Services,"Dublin, Ireland",2011-07-06,1467373,1989
...,...,...,...,...,...,...,...,...
498,XYL,Xylem Inc.,Industrials,Industrial Machinery & Supplies & Components,"White Plains, New York",2011-11-01,1524472,2011
499,YUM,Yum! Brands,Consumer Discretionary,Restaurants,"Louisville, Kentucky",1997-10-06,1041061,1997
500,ZBRA,Zebra Technologies,Information Technology,Electronic Equipment & Instruments,"Lincolnshire, Illinois",2019-12-23,877212,1969
501,ZBH,Zimmer Biomet,Health Care,Health Care Equipment,"Warsaw, Indiana",2001-08-07,1136869,1927


**Ingresamos mercado**

In [4]:
# ---------------------------
# 3️Insertar/actualizar mercados
# ---------------------------
markets = pd.DataFrame({
    'market_name': ['S&P 500'],
    'country': ['USA'],
    'currency': ['USD']
})
markets

,market_name,country,currency
0,S&P 500,USA,USD


In [5]:
db.insert_to_db(markets, tabla="markets", batch_size=5000)

✅ Conexión exitosa


In [6]:
# Obtener market_id
market_id = db.execute_query("SELECT * FROM markets")
market_id

,market_id,market_name,country,currency
0,1,NASDAQ,USA,USD
1,2,S&P 500,USA,USD


**Ingresamos compañias**

In [10]:
market_id = 2

In [11]:

df.loc[:,"market_id"] = [market_id for x in df['Symbol']]
companies = df[["market_id",'Symbol','Security','GICS Sector','GICS Sub-Industry','Date added','Headquarters Location','CIK','Founded']]
companies.columns = ["market_id",'symbol','name','sector_name','sub_industry','date_added','headquarters','cik','founded']
companies = companies.reset_index(drop=True)
companies.head()

,market_id,symbol,name,sector_name,sub_industry,date_added,headquarters,cik,founded
0,2,MMM,3M,Industrials,Industrial Conglomerates,1957-03-04,"Saint Paul, Minnesota",66740,1902
1,2,AOS,A. O. Smith,Industrials,Building Products,2017-07-26,"Milwaukee, Wisconsin",91142,1916
2,2,ABT,Abbott Laboratories,Health Care,Health Care Equipment,1957-03-04,"North Chicago, Illinois",1800,1888
3,2,ABBV,AbbVie,Health Care,Biotechnology,2012-12-31,"North Chicago, Illinois",1551152,2013 (1888)
4,2,ACN,Accenture,Information Technology,IT Consulting & Other Services,2011-07-06,"Dublin, Ireland",1467373,1989


In [12]:
db.insert_to_db(companies, tabla="companies", batch_size=100)

In [13]:
# Obtener mapping symbol -> company_id
company_map = db.execute_query("SELECT company_id, symbol FROM companies")
company_map

,company_id,symbol
0,1,AACB
1,2,AACBR
2,3,AACBU
3,4,AACG
4,5,AACI
...,...,...
5644,5645,XYL
5645,5646,YUM
5646,5647,ZBRA
5647,5648,ZBH


In [14]:
db.close()